In [13]:
%%configure -f
{"defaultLakehouse": {"name": "lh_fraud"}}

StatementMeta(, 4743d727-2aa6-429e-a41d-e2b619dee92e, -1, Finished, Available, Finished, True)

# 02 · Score: rules + 4 models (HGB, LightGBM, LambdaRank, blend), champion picked on Jun–Jul 2026, compared with the legacy engine
Outputs: gold_scores, gold_alert_queue, gold_alert_explanations, gold_model_metrics, gold_model_leaderboard,
gold_model_importance, gold_model_recall_by_type, Files/submission/submission_csv

In [14]:
import json
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
import mlflow
mlflow.autolog(disable=True)
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("lightgbm missing -> add a cell '%pip install lightgbm' after cell 1, or continue with HGB only")

spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
LEGIT_FRAC, SEED = 0.10, 42

g = spark.table("gold_features")
tr = g.filter(F.col("period") == "train")
tr = (tr.filter(F.col("is_fraud") == 1).withColumn("sample_w", F.lit(1.0))
        .unionByName(tr.filter(F.col("is_fraud") == 0).sample(fraction=LEGIT_FRAC, seed=SEED)
                       .withColumn("sample_w", F.lit(1.0 / LEGIT_FRAC))))
rest = g.filter(F.col("period") != "train").withColumn("sample_w", F.lit(1.0))
df = tr.unionByName(rest).toPandas()

fcols = [c for c in df.columns if c.startswith("f_")]
df[fcols] = df[fcols].astype("float32").replace([np.inf, -np.inf], np.nan)
df["is_fraud"] = df["is_fraud"].fillna(0).astype(int)
df = df.sort_values(["period", "timestamp"]).reset_index(drop=True)      # time order (needed by LambdaRank groups)
TR, VA, TE = (df.period == "train").values, (df.period == "valid").values, (df.period == "test").values
print(df.period.value_counts().to_dict(), "| frauds train:", int(df.loc[TR, "is_fraud"].sum()),
      "| frauds valid:", int(df.loc[VA, "is_fraud"].sum()), "|", len(fcols), "features | lightgbm:", HAS_LGB)

StatementMeta(, 10cdb7e2-f75a-43fd-849a-9240b81bb6bc, 3, Finished, Available, Finished, False)

{'test': 187628, 'valid': 177953, 'train': 161647} | frauds train: 8494 | frauds valid: 1013 | 73 features | lightgbm: True


In [15]:
d = df
RULES = [  # (factor, condition, points, observed column, usual column)
    ("Montant très supérieur aux habitudes", (d.f_amount_ratio >= 3) & (d.f_n_before >= 5), 2, "f_amount", "f_avg_amt_before"),
    ("Premier paiement physique dans ce pays", (d.f_new_country_phys == 1) & (d.f_is_foreign == 1) & (d.f_n_before >= 5), 3, "f_amount", None),
    ("Déplacement impossible depuis l'opération précédente", d.f_speed_kmh > 900, 4, "f_speed_kmh", "f_dist_prev_km"),
    ("Rafale de transactions sur la carte (1 h)", d.f_key_tx_1h >= 4, 2, "f_key_tx_1h", None),
    ("Refus récents sur la carte (1 h)", d.f_key_declines_1h >= 2, 2, "f_key_declines_1h", None),
    ("Micro-montants répétés (test de carte)", d.f_key_small_tx_1h >= 3, 2, "f_key_small_tx_1h", None),
    ("Piste magnétique utilisée", d.f_magstripe == 1, 1, None, None),
    ("Opération de nuit inhabituelle pour ce client", d.f_unusual_night == 1, 1, "f_hour", "f_night_share_before"),
    ("Bénéficiaire ajouté moins d'1 h avant le virement", d.f_mins_since_benef_added <= 60, 3, "f_mins_since_benef_added", None),
    ("Échecs de connexion dans les 24 h", d.f_login_fail_24h >= 2, 2, "f_login_fail_24h", None),
    ("Nouvel appareil enrôlé dans les 24 h", (d.f_device_added_24h >= 1) & (d.f_new_device == 1), 1, "f_device_added_24h", None),
    ("Numéro de téléphone modifié dans les 24 h", d.f_phone_change_24h >= 1, 2, "f_phone_change_24h", None),
    ("Sans contact juste sous le plafond chez un nouveau commerçant", (d.f_contactless_near_limit == 1) & (d.f_new_merchant == 1), 2, "f_amount", None),
    ("Commerçant lié à des fraudes signalées (30 j)", d.f_merchant_frauds_30d >= 2, 2, "f_merchant_frauds_30d", None),
    ("Appareil lié à des fraudes signalées (30 j)", d.f_device_frauds_30d >= 1, 3, "f_device_frauds_30d", None),
    ("Carte nationale utilisée à l'étranger", d.f_card_not_intl_abroad == 1, 3, None, None),
    ("Rassurant : carte à puce à l'étranger", d.f_chip_abroad == 1, -1, None, None),
    ("Rassurant : séjour cohérent (≥ 3 opérations dans ce pays sur 7 j)", (d.f_is_foreign == 1) & (d.f_same_country_7d >= 3), -2, "f_same_country_7d", None),
    ("Rassurant : événements sensibles faits en agence", d.f_branch_events_24h >= 1, -1, "f_branch_events_24h", None),
]
hits = np.column_stack([r[1].fillna(False).values.astype(bool) for r in RULES])
points = np.array([r[2] for r in RULES])
df["score_rules"] = (hits * points).sum(axis=1)
print("rules PR-AUC on valid:", round(average_precision_score(df.loc[VA, "is_fraud"], df.loc[VA, "score_rules"]), 4))

StatementMeta(, 10cdb7e2-f75a-43fd-849a-9240b81bb6bc, 4, Finished, Available, Finished, False)

rules PR-AUC on valid: 0.0143


In [16]:
CATS = ["channel", "entry_mode", "transaction_type", "merchant_category", "segment", "kyc_risk_level",
        "card_type", "merchant_type", "decline_reason"]
LEAKY = {"f_merchant_risk", "f_device_risk", "f_beneficiary_risk"}   # target-encoded on all train labels (not point-in-time)
FEATS = [c for c in fcols if c not in LEAKY] + CATS
assert not any(k in c for c in FEATS for k in ["is_fraud", "scenario", "reported", "gender"]), "leak column in features"
X = df[FEATS].copy()
for c in CATS:
    X[c] = pd.Categorical(X[c].astype("string"))               # LightGBM native categoricals
Xh = X.copy()
for c in CATS:
    Xh[c] = X[c].cat.codes.replace(-1, np.nan).astype("float32")  # HGB wants integer codes
cat_mask = [c in CATS for c in FEATS]

def weights(mask):
    y = df.loc[mask, "is_fraud"].values
    sw = df.loc[mask, "sample_w"].values
    ratio = sw[y == 0].sum() / max(y.sum(), 1)                 # real imbalance (sampling undone)
    return y, sw * np.where(y == 1, min(ratio, 50.0), 1.0)

LGB_P = dict(n_estimators=600, learning_rate=0.03, num_leaves=63, min_child_samples=50, subsample=0.8,
             subsample_freq=1, colsample_bytree=0.7, reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbose=-1)

def fit_all(mask):
    y, w = weights(mask)
    m = {"hgb": HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=40,
                                               l2_regularization=1.0, categorical_features=cat_mask,
                                               early_stopping=False, random_state=SEED).fit(Xh[mask], y, sample_weight=w)}
    if HAS_LGB:
        m["lightgbm"] = lgb.LGBMClassifier(**LGB_P).fit(X[mask], y, sample_weight=w)
        day = df.loc[mask, "timestamp"].dt.floor("D")           # one ranking query per day = analyst's daily queue
        grp = day.groupby(day, sort=False).size().values
        m["lambdarank"] = lgb.LGBMRanker(objective="lambdarank", **{**LGB_P, "n_estimators": 400}).fit(X[mask], y, group=grp)
    return m

rk = lambda s: rankdata(s) / len(s)

def score_all(models, mask):
    s = {"hgb": models["hgb"].predict_proba(Xh[mask])[:, 1]}
    if HAS_LGB:
        s["lightgbm"] = models["lightgbm"].predict_proba(X[mask])[:, 1]
        s["lambdarank"] = models["lambdarank"].predict(X[mask])
        # Blend of three model scores only (rules are for explanations, not ranking)
        s["blend"] = (rk(s["hgb"]) + rk(s["lightgbm"]) + rk(s["lambdarank"])) / 3
    return s

models_tr = fit_all(TR)
S_va = score_all(models_tr, VA)
S_va["rules only"] = df.loc[VA, "score_rules"].values

legacy = spark.table("legacy_rule_alerts_train").select("transaction_id").distinct().toPandas()
df["legacy_alert"] = df.transaction_id.isin(set(legacy.transaction_id)).astype(int)
y = df.loc[VA, "is_fraud"].values
k1 = max(int(VA.sum() * 0.01), 1)
k_leg = int(df.loc[VA, "legacy_alert"].sum())

def at_k(score, k):
    idx = np.argsort(-score, kind="stable")[:k]
    tp = int(y[idx].sum())
    return round(tp / max(k, 1), 3), round(tp / max(y.sum(), 1), 3)

rows = []
for name, s in S_va.items():
    p1, r1 = at_k(s, k1)
    pl, rl = at_k(s, k_leg) if k_leg else (np.nan, np.nan)
    rows.append({"model": name, "pr_auc": round(average_precision_score(y, s), 4), "top1pct_precision": p1,
                 "top1pct_recall": r1, "legacy_budget_precision": pl, "legacy_budget_recall": rl})
board = pd.DataFrame(rows).sort_values(["top1pct_recall", "pr_auc"], ascending=False).reset_index(drop=True)
WINNER = board[board.model != "rules only"].iloc[0].model

# prefer LightGBM when within 1 pt of best recall: one model both scores and explains (SHAP)
best = board[board.model != "rules only"].top1pct_recall.max()
if HAS_LGB and board.loc[board.model == "lightgbm", "top1pct_recall"].iloc[0] >= best - 0.01:
    WINNER = "lightgbm"

board["champion"] = (board.model == WINNER).astype(int)
print(board.to_string(index=False))
print("\nCHAMPION:", WINNER)

StatementMeta(, 10cdb7e2-f75a-43fd-849a-9240b81bb6bc, 5, Finished, Available, Finished, False)

     model  pr_auc  top1pct_precision  top1pct_recall  legacy_budget_precision  legacy_budget_recall  champion
  lightgbm  0.6459              0.404           0.709                    0.236                 0.777         1
     blend  0.6543              0.403           0.708                    0.238                 0.783         0
       hgb  0.6398              0.399           0.701                    0.234                 0.772         0
lambdarank  0.6472              0.399           0.700                    0.237                 0.782         0
rules only  0.0143              0.028           0.049                    0.031                 0.101         0

CHAMPION: lightgbm


In [17]:
df["score_model"] = np.nan
df.loc[VA, "score_model"] = S_va[WINNER]
v = df[VA]
M = {"valid_rows": len(v), "valid_frauds": int(y.sum()),
     "pr_auc_model": round(average_precision_score(y, v.score_model), 4),
     "pr_auc_rules": round(average_precision_score(y, v.score_rules), 4),
     "legacy_alerts": k_leg,
     "legacy_precision": round(float(v.loc[v.legacy_alert == 1, "is_fraud"].mean()), 3) if k_leg else 0.0,
     "legacy_recall": round(float(v.loc[v.is_fraud == 1, "legacy_alert"].mean()), 3)}
for pct in [0.005, 0.01, 0.02]:
    k = max(int(len(v) * pct), 1)
    tag = f"top{str(pct * 100).rstrip('0').rstrip('.')}pct"
    M[f"model_{tag}_precision"], M[f"model_{tag}_recall"] = at_k(v.score_model.values, k)
if k_leg:
    M["model_same_budget_as_legacy_precision"], M["model_same_budget_as_legacy_recall"] = at_k(v.score_model.values, k_leg)
    M["frauds_caught_legacy"] = int(round(M["legacy_recall"] * y.sum()))
    M["frauds_caught_model_same_budget"] = int(round(M["model_same_budget_as_legacy_recall"] * y.sum()))
print("CHAMPION:", WINNER)
print(json.dumps(M, indent=1))

top = v.nlargest(k1, "score_model")
per_type = (v[v.is_fraud == 1].groupby("fraud_scenario").transaction_id.count().to_frame("frauds")
             .join(top[top.is_fraud == 1].groupby("fraud_scenario").transaction_id.count().to_frame("caught_top1pct"))
             .join(v[(v.is_fraud == 1) & (v.legacy_alert == 1)].groupby("fraud_scenario").transaction_id.count().to_frame("caught_legacy"))
             .fillna(0).astype(int))
per_type["recall_model"] = (per_type.caught_top1pct / per_type.frauds).round(3)
per_type["recall_legacy"] = (per_type.caught_legacy / per_type.frauds).round(3)
print(per_type.sort_values("frauds", ascending=False).to_string())

if HAS_LGB:
    imp = pd.Series(models_tr["lightgbm"].booster_.feature_importance("gain"), index=FEATS)
    importance = (imp / imp.sum()).round(4).nlargest(20).rename("gain_share").reset_index().rename(columns={"index": "feature"})
    print("\nTop drivers:\n", importance.head(15).to_string(index=False))

StatementMeta(, 10cdb7e2-f75a-43fd-849a-9240b81bb6bc, 6, Finished, Available, Finished, False)

CHAMPION: lightgbm
{
 "valid_rows": 177953,
 "valid_frauds": 1013,
 "pr_auc_model": 0.6459,
 "pr_auc_rules": 0.0143,
 "legacy_alerts": 3338,
 "legacy_precision": 0.035,
 "legacy_recall": 0.115,
 "model_top0.5pct_precision": 0.664,
 "model_top0.5pct_recall": 0.582,
 "model_top1pct_precision": 0.404,
 "model_top1pct_recall": 0.709,
 "model_top2pct_precision": 0.223,
 "model_top2pct_recall": 0.782,
 "model_same_budget_as_legacy_precision": 0.236,
 "model_same_budget_as_legacy_recall": 0.777,
 "frauds_caught_legacy": 116,
 "frauds_caught_model_same_budget": 787
}
                          frauds  caught_top1pct  caught_legacy  recall_model  recall_legacy
fraud_scenario                                                                              
CARTE_VOLEE_SANS_CONTACT     406             275              0         0.677          0.000
FRAUDE_EN_LIGNE_DISCRETE     264             180             33         0.682          0.125
SKIMMING_DAB                 182             134             3

In [18]:
models_full = fit_all(TR | VA)
S_te = score_all(models_full, TE)
df.loc[TE, "score_model"] = S_te[WINNER]

# scored = all non-train rows (valid + test)
scored = ~TR

# Source of the score (for later analysis and app display)
df["score_source"] = np.select([TR, VA, TE], ["in_sample", "validation", "test"], "other")

# Rank within each period, highest score = rank 1
df["rank_in_period"] = 0
df.loc[scored, "rank_in_period"] = (
    df[scored]
      .groupby("period")["score_model"]
      .rank(ascending=False, method="first")
      .astype(int)
)

# Alert budget = top 1% per period, based on all rows of that period
# (valid and test are fully scored; train rows are never marked as alerts)
n_per = df.groupby("period").transaction_id.transform("size")
budget = (n_per * 0.01).astype(int).clip(lower=1)

# Percentile score within period (1.0 = riskiest in that period)
df["score_pct"] = np.nan
df.loc[scored, "score_pct"] = (
    1 - (df.loc[scored, "rank_in_period"] - 1) / n_per.loc[scored]
).round(5)

# Alerts and risk levels
df["is_alert"] = 0
df.loc[scored, "is_alert"] = (
    df.loc[scored, "rank_in_period"] <= budget.loc[scored]
).astype(int)

df["risk_level"] = "FAIBLE"
df.loc[scored & (df.is_alert == 1), "risk_level"] = "MOYEN"
df.loc[scored & (df.is_alert == 1) & (df.rank_in_period <= budget * 0.3), "risk_level"] = "ELEVE"

print("alerts:", df[scored].groupby("period").is_alert.sum().to_dict())

StatementMeta(, 10cdb7e2-f75a-43fd-849a-9240b81bb6bc, 7, Finished, Available, Finished, False)

alerts: {'test': 1876, 'valid': 1779}


In [ ]:
va = df[VA]
obvious = va[va.is_fraud == 1].nlargest(1, "score_model").transaction_id.tolist()
subtle = va[(va.is_fraud == 1) & (va.is_alert == 1) & (va.score_rules <= 1)].nlargest(1, "score_model").transaction_id.tolist()
avoided_fp = va[(va.legacy_alert == 1) & (va.is_fraud == 0)].nsmallest(1, "score_model").transaction_id.tolist()
DEMO = {"obvious_fraud": obvious, "subtle_fraud": subtle, "avoided_false_positive": avoided_fp}
DEMO_IDS = set(obvious + subtle + avoided_fp)
print("DEMO:", DEMO)

LABEL = {
 "f_amount": "Montant de l'opération", "f_amount_ratio": "Montant / moyenne habituelle", "f_amount_z": "Écart aux habitudes (z-score)",
 "f_amount_over_max": "Montant / plus gros montant passé", "f_amount_over_income": "Montant / revenu mensuel",
 "f_amount_over_limit": "Montant / plafond carte", "f_key_tx_10m": "Opérations carte (10 min avant)",
 "f_key_tx_1h": "Opérations carte (1 h avant)", "f_key_declines_1h": "Refus carte (1 h avant)",
 "f_key_small_tx_1h": "Micro-montants carte (1 h avant)", "f_cust_tx_24h": "Opérations client (24 h avant)",
 "f_cust_amt_24h": "Montant cumulé client (24 h avant)", "f_new_merchant": "Nouveau commerçant", "f_new_device": "Nouvel appareil",
 "f_new_beneficiary": "Nouveau bénéficiaire", "f_new_ip_country": "Nouveau pays d'IP", "f_new_country_phys": "Premier paiement physique dans ce pays",
 "f_new_city_phys": "Première opération physique dans cette ville", "f_is_foreign": "Opération à l'étranger", "f_ip_foreign": "IP étrangère",
 "f_speed_kmh": "Vitesse implicite depuis l'opération précédente (km/h)", "f_dist_prev_km": "Distance depuis l'opération précédente (km)",
 "f_secs_since_prev": "Secondes depuis l'opération précédente", "f_hour": "Heure", "f_unusual_night": "Nuit inhabituelle pour ce client",
 "f_magstripe": "Piste magnétique", "f_contactless": "Sans contact", "f_ecom_no3ds": "E-commerce sans 3-D Secure",
 "f_card_not_intl_abroad": "Carte nationale à l'étranger", "f_login_fail_24h": "Échecs de connexion (24 h)",
 "f_device_added_24h": "Appareil ajouté (24 h)", "f_benef_added_24h": "Bénéficiaire ajouté (24 h)",
 "f_mins_since_benef_added": "Minutes depuis l'ajout du bénéficiaire", "f_phone_change_24h": "Changement de numéro", "f_pwd_reset_24h": "Réinitialisation mot de passe (24 h)",
 "f_merchant_frauds_30d": "Fraudes signalées chez ce commerçant (30 j)",
 "f_device_frauds_30d": "Fraudes signalées sur cet appareil (30 j)", "f_n_before": "Historique client (nb d'opérations)",
 "f_declined": "Opération refusée", "f_same_country_7d": "Opérations dans ce pays (7 j)", "f_chip_abroad": "Carte à puce à l'étranger",
 "f_branch_events_24h": "Événements en agence (24 h)", "channel": "Canal", "entry_mode": "Mode de saisie",
 "transaction_type": "Type d'opération", "merchant_category": "Catégorie commerçant", "decline_reason": "Motif de refus",
 "segment": "Segment client", "kyc_risk_level": "Niveau de risque KYC", "card_type": "Type de carte", "merchant_type": "Type de commerçant",
 "f_mins_since_pwd_reset": "Minutes depuis la réinitialisation du mot de passe",
 "f_mins_since_phone_change": "Minutes depuis le changement de numéro",
 "f_pattern_pwd_reset_tx_60m": "Mot de passe réinitialisé < 60 min avant",
 "f_pattern_phone_change_tx_120m": "Numéro changé < 120 min avant",
 "f_merchant_fraud_customers_30d": "Clients fraudés chez ce commerçant (30 j)",
 "f_device_fraud_customers_30d": "Clients fraudés sur cet appareil (30 j)",
 "f_merchant_fraud_cards_30d": "Cartes fraudées chez ce commerçant (30 j)",
 "f_device_fraud_cards_30d": "Cartes fraudées sur cet appareil (30 j)",
 "f_new_category": "Nouvelle catégorie de commerçant pour ce client",
 "f_key_contactless_24h": "Paiements sans contact sur la carte (24 h)",
 "f_key_new_merchants_24h": "Nouveaux commerçants sur la carte (24 h)",
 "f_round_amount": "Montant rond",
 "f_benef_n_customers_before": "Clients ayant déjà payé ce bénéficiaire"}

expl_mask = (scored & ((df.is_alert == 1) | df.transaction_id.isin(DEMO_IDS))).values
rows = []
for i in np.where(expl_mask)[0]:
    r = df.iloc[i]
    for j in sorted(np.where(hits[i])[0], key=lambda j: -abs(points[j]))[:5]:
        name, _, pts, obs, usual = RULES[j]
        rows.append({"transaction_id": r.transaction_id, "factor": name, "feature": None, "points": float(pts),
                     "observed": None if obs is None or pd.isna(r[obs]) else round(float(r[obs]), 3),
                     "usual": None if usual is None or pd.isna(r[usual]) else round(float(r[usual]), 3),
                     "observed_text": None, "source": "rules"})

if HAS_LGB:   # exact per-transaction SHAP from LightGBM (no extra package); valid explained by the train model, test by the refit
    for per, mdl in [("valid", models_tr["lightgbm"]), ("test", models_full["lightgbm"])]:
        m = expl_mask & (df.period.values == per)
        if not m.any():
            continue
        contrib = mdl.predict(X[m], pred_contrib=True)[:, :-1]
        ids, vals = df.loc[m, "transaction_id"].values, X[m].astype(object).to_numpy()
        avg = df.loc[m, "f_avg_amt_before"].values
        for i in range(len(ids)):
            order = np.argsort(-contrib[i])
            picks = [(j, "model_for") for j in order[:3] if contrib[i, j] > 0] + \
                    [(j, "model_against") for j in order[::-1][:2] if contrib[i, j] < 0]
            for j, src in picks:
                feat, val = FEATS[j], vals[i, j]
                num = feat not in CATS and val is not None and not pd.isna(val)
                rows.append({"transaction_id": ids[i], "factor": LABEL.get(feat, feat), "feature": feat,
                             "points": round(float(contrib[i, j]), 4),
                             "observed": round(float(val), 3) if num else None,
                             "usual": round(float(avg[i]), 3) if feat == "f_amount" and not pd.isna(avg[i]) else None,
                             "observed_text": None if num or val is None or pd.isna(val) else str(val), "source": src})

expl = pd.DataFrame(rows, columns=["transaction_id", "factor", "feature", "points", "observed", "usual", "observed_text", "source"])
for c in ["points", "observed", "usual"]:
    expl[c] = pd.to_numeric(expl[c], errors="coerce").astype(float)
print(len(expl), "explanation rows for", int(expl_mask.sum()), "transactions |", expl.source.value_counts().to_dict())

StatementMeta(, 10cdb7e2-f75a-43fd-849a-9240b81bb6bc, 8, Finished, Available, Finished, False)

DEMO: {'obvious_fraud': ['TX01555470'], 'subtle_fraud': ['TX01585580'], 'avoided_false_positive': ['TX01575959']}


20145 explanation rows for 3656 transactions | {'model_for': 10968, 'model_against': 7312, 'rules': 1865}


In [ ]:
def save(pdf, name, null_labels=False):
    pdf = pdf.copy()
    for c in pdf.select_dtypes("object").columns:
        pdf[c] = pdf[c].where(pdf[c].notna(), None)
        if pdf[c].isna().all():
            pdf[c] = pdf[c].astype("float64")       # avoid NullType schema errors on empty columns
    sdf = spark.createDataFrame(pdf)
    if null_labels:
        is_test = F.col("period") == "test"
        sdf = (sdf.withColumn("is_fraud", F.when(~is_test, F.col("is_fraud")).cast("int"))
                  .withColumn("fraud_scenario", F.when(~is_test, F.col("fraud_scenario"))))
    sdf.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(name)

save(df.loc[scored, ["transaction_id", "period", "score_source", "score_model", "score_pct", "score_rules",
                     "rank_in_period", "is_alert", "risk_level", "legacy_alert"]], "gold_scores")

top_rules = (expl[expl.source == "rules"].groupby("transaction_id").factor.apply(lambda s: " | ".join(s.head(3))).rename("top_factors"))
top_model = (expl[expl.source == "model_for"].groupby("transaction_id").factor.apply(lambda s: " | ".join(s.head(3))).rename("top_model_factors"))
queue_cols = ["transaction_id", "timestamp", "period", "customer_id", "card_id", "transaction_type", "channel",
              "entry_mode", "merchant_category", "amount_tnd", "city", "country", "status", "score_model", "score_pct",
              "score_rules", "rank_in_period", "risk_level", "legacy_alert", "is_fraud", "fraud_scenario"]
queue = (df.loc[expl_mask, queue_cols].merge(top_rules, on="transaction_id", how="left")
                                      .merge(top_model, on="transaction_id", how="left"))
queue["demo_case"] = queue.transaction_id.map({t: k for k, ids in DEMO.items() for t in ids})
save(queue, "gold_alert_queue", null_labels=True)
save(expl, "gold_alert_explanations")
save(pd.DataFrame([{"metric": k, "value": float(val)} for k, val in M.items()]), "gold_model_metrics")
save(board, "gold_model_leaderboard")
save(per_type.reset_index(), "gold_model_recall_by_type")
if HAS_LGB:
    save(importance, "gold_model_importance")

submission = df.loc[TE, ["transaction_id", "score_model", "is_alert"]].rename(columns={"score_model": "fraud_score"})
(spark.createDataFrame(submission).coalesce(1).write.mode("overwrite").option("header", True)
      .csv("Files/submission/submission_csv"))
print("written:", len(queue), "queue rows |", len(expl), "explanation rows |", len(submission), "submission rows | champion:", WINNER)

StatementMeta(, 10cdb7e2-f75a-43fd-849a-9240b81bb6bc, 9, Finished, Available, Finished, False)

written: 3656 queue rows | 20145 explanation rows | 187628 submission rows | champion: lightgbm


In [ ]:
print("=== CHECK 02 ===  champion:", WINNER)
for t in ["gold_scores", "gold_alert_queue", "gold_alert_explanations", "gold_model_metrics",
          "gold_model_leaderboard", "gold_model_recall_by_type"]:
    print(f"{t:28s}", spark.table(t).count())
q = spark.table("gold_alert_queue")
q.groupBy("period", "risk_level").count().orderBy("period", "risk_level").show()
print("test rows with a label in queue (must be 0):", q.filter((F.col("period") == "test") & F.col("is_fraud").isNotNull()).count())
q.filter(F.col("demo_case").isNotNull()).select("demo_case", "transaction_id", "amount_tnd", "channel", "country",
                                                "score_pct", "top_factors", "top_model_factors").show(truncate=False)

StatementMeta(, 10cdb7e2-f75a-43fd-849a-9240b81bb6bc, 10, Finished, Available, Finished, False)

=== CHECK 02 ===  champion: lightgbm


gold_scores                  365581


gold_alert_queue             3656


gold_alert_explanations      20145


gold_model_metrics           17


gold_model_leaderboard       5


gold_model_recall_by_type    6


+------+----------+-----+
|period|risk_level|count|
+------+----------+-----+
|  test|     ELEVE|  562|
|  test|     MOYEN| 1314|
| valid|     ELEVE|  533|
| valid|    FAIBLE|    1|
| valid|     MOYEN| 1246|
+------+----------+-----+



test rows with a label in queue (must be 0): 0
+----------------------+--------------+----------+--------+-------+---------+------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------+
|demo_case             |transaction_id|amount_tnd|channel |country|score_pct|top_factors                                                                                                             |top_model_factors                                                                         |
+----------------------+--------------+----------+--------+-------+---------+------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------+
|subtle_fraud          |TX01585580    |7.685     |EN_LIGNE|TN     |0.99993  |Opérat

In [ ]:
import pandas as pd

wrangler_sample_df = pd.read_csv("https://aka.ms/wrangler/titanic.csv")
display(wrangler_sample_df)

StatementMeta(, 0cb1c4d6-4e72-467d-b0a8-518f4253236f, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5b317e61-096f-4619-be64-e737891b5895)

In [ ]:
import pandas as pd

wrangler_sample_df = pd.read_csv("https://aka.ms/wrangler/titanic.csv")
display(wrangler_sample_df)

StatementMeta(, , -1, Finished, , Finished, True)

Cancelled: Session has been cancelled.

In [ ]:
for t in ["gold_alert_queue", "gold_alert_explanations", "gold_model_metrics",
          "gold_model_leaderboard", "gold_model_recall_by_type", "gold_model_importance"]:
    spark.table(t).coalesce(1).write.mode("overwrite").parquet(f"Files/app_export/{t}")
print("exported for app")

StatementMeta(, 10cdb7e2-f75a-43fd-849a-9240b81bb6bc, 11, Finished, Available, Finished, False)

exported for app


In [56]:
from pyspark.sql import functions as F

q_cust = spark.table("gold_alert_queue").select("customer_id").distinct()

hist_cols = [
    "transaction_id", "timestamp", "period", "customer_id", "card_id", "merchant_id", "beneficiary_id", "device_id",
    "transaction_type", "channel", "entry_mode", "merchant_category", "amount_tnd", "city", "country",
    "ip_country", "status", "decline_reason"
]

(
    spark.table("gold_features")
         .filter(F.col("period") != "train")
         .join(q_cust, "customer_id")
         .select(*hist_cols)
         .coalesce(1)
         .write.mode("overwrite")
         .parquet("Files/app_export/customer_history")
)

(
    spark.table("customers")
         .drop("gender")
         .join(q_cust, "customer_id")
         .coalesce(1)
         .write.mode("overwrite")
         .parquet("Files/app_export/customers")
)

(
    spark.table("digital_events")
         .join(q_cust, "customer_id")
         .filter(F.col("timestamp") >= "2026-05-01")
         .coalesce(1)
         .write.mode("overwrite")
         .parquet("Files/app_export/digital_events")
)

for t in ["customer_history", "customers", "digital_events"]:
    cnt = spark.read.parquet(f"Files/app_export/{t}").count()
    print(t, cnt)

StatementMeta(, 4743d727-2aa6-429e-a41d-e2b619dee92e, 5, Finished, Available, Finished, False)

customer_history 193540
customers 1814
digital_events 97376


In [ ]:
mssparkutils.session.stop()   # free the capacity

StatementMeta(, 10cdb7e2-f75a-43fd-849a-9240b81bb6bc, 12, Finished, Available, Finished, False)